## UDA-Hub：CultPass のカスタマーサポート

問い合わせ（チケット）を受け取り、Classifier が振り分けて、Knowledge／Account／Escalation のどれかが対応する。

```text
load_memory → classifier ─┬─ knowledge ──────────────┬─→ save_memory
                          ├─ account ⇄ account_tools ─┤
                          └─ escalation ←─────────────┘
```

- 短期メモリ：`MemorySaver`（`thread_id` = `ticket_id`）
- 長期メモリ：udahub.db（`ticket_messages`、`ticket_metadata`）
- ログ：`logs/workflow.jsonl`（JSON Lines）

**注意**：このノートブックは udahub.db にデモのチケットを書き込む。元に戻すには `02_core_db_setup.ipynb` を実行し直す。

## Setup

In [ ]:
from dotenv import load_dotenv
from utils import chat_interface

load_dotenv()

In [ ]:
from agentic.workflow import orchestrator, new_ticket

### 1 ターン実行するための関数

`chat_interface` と同じ形（`messages` と `thread_id` だけ）でグラフを呼ぶ。`user_id` は、グラフの `load_memory` が `ticket_id` から udahub.db を引いて知る。

In [ ]:
from langchain_core.messages import HumanMessage


def ask(ticket_id: str, text: str) -> dict:
    config = {"configurable": {"thread_id": ticket_id}}
    result = orchestrator.invoke({"messages": [HumanMessage(content=text)]}, config=config)
    print(f"User: {text}")
    print(f"  -> category={result['category']}, urgency={result['urgency']}, "
          f"resolution={result['resolution']}")
    if result.get("escalation_reason"):
        print(f"  -> escalation_reason: {result['escalation_reason']}")
    print(f"Assistant: {result['messages'][-1].content}\n")
    return result

## Run

### シナリオ 1：ナレッジ記事で解決する（Knowledge）

In [ ]:
ticket_1 = new_ticket("f1f10d")
result = ask(ticket_1, "The QR code won't scan at the museum entrance")
print("Retrieved articles:", result["retrieved_articles"])

### シナリオ 2：ツールで CultPass DB を照会する（Account）＋ 同じチケットで会話を続ける（短期メモリ）

In [ ]:
ticket_2 = new_ticket("f1f10d")
ask(ticket_2, "Is my subscription still active?")
ask(ticket_2, "And how many experiences can I use per month on that plan?");

### シナリオ 3：記事が人への引き継ぎを指示している（Knowledge → Escalation）＋ 長期メモリ

同じユーザー（`f1f10d`）の新しいチケット。`past_context` に、シナリオ 1、2 のチケットが出る。

In [ ]:
ticket_3 = new_ticket("f1f10d")
result = ask(ticket_3, "I was charged twice this month")
print("Past context:\n" + result["past_context"])
print("\nHandoff summary:\n" + result["handoff_summary"])

### シナリオ 4：ブロック済みのユーザーが予約のキャンセルを依頼する（Account → Escalation）

In [ ]:
ticket_4 = new_ticket("a4ab87")
ask(ticket_4, "Please cancel my booking for next week");

### シナリオ 5：記事にない質問（信頼度が低い）と、人を求める問い合わせ（Escalation）

In [ ]:
ticket_5 = new_ticket("88382b")
ask(ticket_5, "Do you sell gift cards?")

ticket_6 = new_ticket("88382b")
ask(ticket_6, "責任者と話をさせてください。もう我慢できません。");

## 確認

### 短期メモリ：チケットの State（ツールの呼び出しも含む）

In [ ]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": ticket_2,
        }
    }
))[0].values["messages"]

### 長期メモリ：udahub.db に保存されたチケットの結果

In [ ]:
from agentic.tools.memory import get_ticket_info

for t in [ticket_1, ticket_2, ticket_3, ticket_4, ticket_5, ticket_6]:
    info = get_ticket_info(t)["data"]
    print(info["external_user_id"], info["status"], info["main_issue_type"], info["tags"])

### ログ（JSON Lines）：シナリオ 3 の流れ

In [ ]:
from agentic.logging_utils import read_logs

for record in read_logs(ticket_3):
    print(record)

## 対話モード

`quit` で終了する。

In [ ]:
chat_interface(orchestrator, new_ticket("f1f10d"))